# 2. Process PyNNLF Output: Ausgrid Solar Home Dataset (ASHD) 148hh Forecast Horizons

## Purpose
Creates ASHD-only tables for the three forecast horizons.

## Inputs
- `RECAP_PATH`

## Run Flow
1. Setup And Paths
2. Load And Validate ASHD Results
3. Build Horizon Tables

## Outputs
- `RESULTS_DIR / "ashd_148hh_horizon_combined_recap.csv"`
- `RESULTS_DIR / "ashd_148hh_weather_nrmse_by_horizon.csv"`
- `RESULTS_DIR / "ashd_148hh_weather_nrmse_stddev_by_horizon.csv"`
- `RESULTS_DIR / "paper_table_ashd_148hh_horizon_test_nrmse.csv"`
- `RESULTS_DIR / "paper_table_ashd_148hh_horizon_key_models_train_test_runtime.csv"`

## 1. Setup And Paths

In [1]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
RESULTS_DIR = PROJECT_DIR / "results" / "02_ashd_148hh_forecast_horizon"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RECAP_PATH = PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv"
MODEL_ORDER = ['m1_naive_hp1', 'm2_snaive_hp2', 'm3_ets_hp1', 'm4_arima_hp1', 'm6_lr_hp1', 'm7_ann_hp1', 'm8_dnn_hp1', 'm9_rt_hp3', 'm10_rf_hp1', 'm13_lstm_hp2', 'm16_prophet_hp1', 'm17_xgb_hp1']
HORIZON_ORDER = [30, 1440, 10080]
HORIZON_LABELS = {30: "30_min", 1440: "1_day", 10080: "1_week"}

Publication project: <local path redacted>
Repository root: <local path redacted>


## 2. Load And Validate ASHD Results

In [2]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(f"Missing recap at {RECAP_PATH}. Run notebook 1 first.")
recap = pd.read_csv(RECAP_PATH)
required = {"dataset_no", "forecast_horizon_min", "model_name", "train_nRMSE", "test_nRMSE", "test_nRMSE_stddev", "runtime_ms"}
missing = required - set(recap.columns)
if missing:
    raise ValueError(f"Recap is missing required columns: {sorted(missing)}")
ashd = recap.loc[recap["dataset_no"].eq("ds20") & pd.to_numeric(recap["forecast_horizon_min"], errors="coerce").isin(HORIZON_ORDER)].copy()
ashd["forecast_horizon_min"] = ashd["forecast_horizon_min"].astype(int)
ashd["model_name"] = pd.Categorical(ashd["model_name"], categories=MODEL_ORDER, ordered=True)
ashd["horizon_label"] = pd.Categorical(ashd["forecast_horizon_min"].map(HORIZON_LABELS), categories=[HORIZON_LABELS[h] for h in HORIZON_ORDER], ordered=True)
ashd["dataset_label"] = "ASHD_148hh_weather"
expected_keys = {(horizon, model) for horizon in HORIZON_ORDER for model in MODEL_ORDER}
actual_keys = set(zip(ashd["forecast_horizon_min"].astype(int), ashd["model_name"].astype(str)))
missing_keys = sorted(expected_keys - actual_keys)
if missing_keys:
    raise ValueError(f"ASHD ds20 results are incomplete. Missing {len(missing_keys)} combinations: {missing_keys[:10]}")
if ashd.shape[0] != len(expected_keys):
    raise ValueError(f"Expected {len(expected_keys)} ASHD rows, found {ashd.shape[0]}")
ashd = ashd.sort_values(["forecast_horizon_min", "model_name"])
display(ashd[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])

   dataset_no  forecast_horizon_min  ... test_nRMSE  test_nRMSE_stddev
36       ds20                    30  ...   4.224911           0.537976
37       ds20                    30  ...  10.946238           0.971774
38       ds20                    30  ...   4.146565           0.506036
39       ds20                    30  ...   3.557113           0.340208
40       ds20                    30  ...   2.657142           0.180678
41       ds20                    30  ...   6.531439           0.898246
42       ds20                    30  ...   4.138044           0.258904
43       ds20                    30  ...   7.451209           0.952487
44       ds20                    30  ...   4.967375           0.527842
45       ds20                    30  ...   9.487459           1.532476
46       ds20                    30  ...   9.174863           1.179247
47       ds20                    30  ...   2.611803           0.242532
48       ds20                  1440  ...   9.091399           0.904674
49    

## 3. Build Horizon Tables

In [3]:
def wide_metric_by_horizon(df, metric_column):
    table = df.pivot_table(index="model_name", columns="horizon_label", values=metric_column, aggfunc="first", observed=False)
    table = table.reindex(index=MODEL_ORDER)
    table = table[[HORIZON_LABELS[h] for h in HORIZON_ORDER]]
    table.index.name = "model_hp"
    table.columns.name = None
    return table

PAPER_MODEL_ORDER = ['m17_xgb_hp1', 'm8_dnn_hp1', 'm6_lr_hp1', 'm10_rf_hp1', 'm7_ann_hp1', 'm1_naive_hp1', 'm3_ets_hp1', 'm9_rt_hp3', 'm13_lstm_hp2', 'm2_snaive_hp2', 'm16_prophet_hp1', 'm4_arima_hp1']
PAPER_KEY_MODELS = ['m17_xgb_hp1', 'm1_naive_hp1', 'm4_arima_hp1']
PAPER_MODEL_LABELS = {'m1_naive_hp1': 'naive_hp1', 'm2_snaive_hp2': 'snaive_hp2', 'm3_ets_hp1': 'ets_hp1', 'm4_arima_hp1': 'arima_hp1', 'm6_lr_hp1': 'lr_hp1', 'm7_ann_hp1': 'ann_hp1', 'm8_dnn_hp1': 'dnn_hp1', 'm9_rt_hp3': 'rt_hp3', 'm10_rf_hp1': 'rf_hp1', 'm13_lstm_hp2': 'lstm_hp2', 'm16_prophet_hp1': 'prophet_hp1', 'm17_xgb_hp1': 'xgb_hp1'}
PAPER_HORIZON_LABELS = {"30_min": "30 minutes", "1_day": "1 day", "1_week": "1 week"}

def short_model_name(model_name):
    return PAPER_MODEL_LABELS.get(str(model_name), str(model_name))

ashd_nrmse = wide_metric_by_horizon(ashd, "test_nRMSE")
ashd_stddev = wide_metric_by_horizon(ashd, "test_nRMSE_stddev")

def get_single_value(frame, horizon_label, model_name, column):
    match = frame.loc[frame["horizon_label"].astype(str).eq(horizon_label) & frame["model_name"].astype(str).eq(model_name), column]
    if match.empty:
        raise ValueError(f"Missing {column} for {horizon_label} / {model_name}")
    return pd.to_numeric(match.iloc[0], errors="coerce")

paper_table4 = (
    ashd_nrmse
    .rename(index=short_model_name, columns=PAPER_HORIZON_LABELS)
    .sort_values("30 minutes", kind="mergesort")
    .round(2)
)

paper_rows = []
for model in PAPER_KEY_MODELS:
    row = {"Model Name": short_model_name(model)}
    for horizon_label in [HORIZON_LABELS[h] for h in HORIZON_ORDER]:
        display_label = PAPER_HORIZON_LABELS[horizon_label]
        row[f"{display_label} Train nRMSE (%)"] = get_single_value(ashd, horizon_label, model, "train_nRMSE")
        row[f"{display_label} Test nRMSE (%)"] = get_single_value(ashd, horizon_label, model, "test_nRMSE")
        row[f"{display_label} Training Time (s)"] = get_single_value(ashd, horizon_label, model, "runtime_ms") / 1000.0
    paper_rows.append(row)
paper_table5 = pd.DataFrame(paper_rows)
for column in paper_table5.columns.drop("Model Name"):
    paper_table5[column] = pd.to_numeric(paper_table5[column], errors="coerce").round(1)

ashd.to_csv(RESULTS_DIR / "ashd_148hh_horizon_combined_recap.csv", index=False)
ashd_nrmse.to_csv(RESULTS_DIR / "ashd_148hh_weather_nrmse_by_horizon.csv")
ashd_stddev.to_csv(RESULTS_DIR / "ashd_148hh_weather_nrmse_stddev_by_horizon.csv")
paper_table4.to_csv(RESULTS_DIR / "paper_table_ashd_148hh_horizon_test_nrmse.csv")
paper_table5.to_csv(RESULTS_DIR / "paper_table_ashd_148hh_horizon_key_models_train_test_runtime.csv", index=False)

display(ashd_nrmse.round(3))
display(ashd_stddev.round(3))
display(paper_table4)
display(paper_table5)

                 30_min   1_day  1_week
model_hp                               
m1_naive_hp1      4.225   9.091  10.937
m2_snaive_hp2    10.946  10.953  10.937
m3_ets_hp1        4.147   9.142  10.981
m4_arima_hp1      3.557  16.179  17.847
m6_lr_hp1         2.657   7.763   8.757
m7_ann_hp1        6.531   8.150   8.836
m8_dnn_hp1        4.138   7.690   8.578
m9_rt_hp3         7.451  10.122  10.445
m10_rf_hp1        4.967   8.490   8.995
m13_lstm_hp2      9.487   9.904  10.114
m16_prophet_hp1   9.175   9.212   9.741
m17_xgb_hp1       2.612   6.087   6.611
                 30_min  1_day  1_week
model_hp                              
m1_naive_hp1      0.538  0.905   0.855
m2_snaive_hp2     0.972  0.972   0.855
m3_ets_hp1        0.506  0.944   0.895
m4_arima_hp1      0.340  1.981   2.402
m6_lr_hp1         0.181  0.676   0.745
m7_ann_hp1        0.898  0.777   0.790
m8_dnn_hp1        0.259  0.671   0.844
m9_rt_hp3         0.952  0.763   0.904
m10_rf_hp1        0.528  0.506   0.653
m13_lstm_hp